## Bronze Layer — Activity Gear
Fetches gear associated with each activity from the Garmin API.
One row per activity-gear combination.
Incremental load based on activities not yet in the gear table.

In [0]:
%pip install garminconnect

In [0]:
import pandas as pd
from garminconnect import Garmin
import json
import time

In [0]:
email = dbutils.secrets.get(scope="garmin", key="email")
password = dbutils.secrets.get(scope="garmin", key="password")
print("Credentials loaded ✅")

In [0]:
api = Garmin(email=email, password=password)
api.login()
print("Garmin API connected ✅")

In [0]:
# Pipeline parameters
# full: fetch gear for all activities
# incremental: only fetch gear for activities not yet in gear table
load_type = "incremental"

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import lit
from pyspark.sql.types import StringType

if load_type == "full":
    # Get all activity IDs from bronze
    activity_ids = [row.activityId for row in spark.table("garmin_lakehouse.raw.activities").select("activityId").collect()]
    
    print(f"Full load: fetching gear for {len(activity_ids)} activities")

else:
    # Get activity IDs not yet in gear table
    all_ids = spark.table("garmin_lakehouse.raw.activities").select("activityId")
    
    # Check if gear table exists
    if spark.catalog.tableExists("garmin_lakehouse.raw.activity_gear"):
        existing_ids = spark.table("garmin_lakehouse.raw.activity_gear").select("activityId")
        new_ids = all_ids.subtract(existing_ids)
    else:
        # Table doesn't exist yet — treat as full load
        print("Gear table not found — running full load")
        new_ids = all_ids

    activity_ids = [row.activityId for row in new_ids.collect()]
    print(f"Incremental load: fetching gear for {len(activity_ids)} activities")

In [0]:
gear_results = []
skipped = 0
fetched = 0

for activity_id in activity_ids:
    gear = api.get_activity_gear(activity_id)
    
    if gear:
        for item in gear:
            item['activityId'] = activity_id
            gear_results.append(item)
        fetched += 1
    else:
        skipped += 1
    
    time.sleep(0.1)

print(f"Activities with gear: {fetched}")
print(f"Activities without gear: {skipped}")
print(f"Total gear records: {len(gear_results)}")

In [0]:
if not gear_results:
    print("No gear data to write — skipping table update")
else:
    # Convert to DataFrame
    df_gear = pd.DataFrame(gear_results)

    # Drop unused image columns
    image_cols = ['imageNameLarge', 'imageNameMedium', 'imageNameSmall']
    df_gear = df_gear.drop(columns=[col for col in image_cols if col in df_gear.columns])

    print(f"Columns: {len(df_gear.columns)}")
    print(f"Rows: {len(df_gear)}")

    # Serialise complex fields
    for col_name in df_gear.columns:
        if df_gear[col_name].apply(lambda x: isinstance(x, (dict, list))).any():
            df_gear[col_name] = df_gear[col_name].apply(
                lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
            )

    # Convert to Spark DataFrame
    spark_df = spark.createDataFrame(df_gear)

    # Write to Delta table
    if not spark.catalog.tableExists("garmin_lakehouse.raw.activity_gear"):
        spark_df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable("garmin_lakehouse.raw.activity_gear")
        print("Table created ✅")
    else:
        from delta.tables import DeltaTable
        
        delta_table = DeltaTable.forName(spark, "garmin_lakehouse.raw.activity_gear")
        
        delta_table.alias("existing") \
            .merge(
                spark_df.alias("new"),
                "existing.activityId = new.activityId AND existing.gearPk = new.gearPk"
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        print("Merge complete ✅")

    # Verify
    count = spark.table("garmin_lakehouse.raw.activity_gear").count()
    print(f"Total rows: {count}")